# Face Recognition Dataset Cleaner (MS1M-ArcFace) - PyTorch / T4 x2

**What this notebook does**
1. Installs the requirements (facenet-pytorch MTCNN + gdown) with zero CUDA setup - plain PyTorch just works on Kaggle GPU.
2. Downloads and extracts `data.zip` from Google Drive by file id.
3. For each identity: detects the face, scores frontal-pose / neutrality / uprightness / sharpness / resolution, and keeps the **2 best** full original images (`reference` + `match`).
4. **Verifies the pair is the same person**: embeds `reference` and `match` with a face-recognition model (InceptionResnetV1, VGGFace2) and keeps the identity only if their **cosine similarity >= 0.50**. If they don't match, the identity is **dropped** and is **not counted** toward the 1000.
5. Runs on **both T4 GPUs** (one process per GPU) and saves a zip **every 1000 matched identities per GPU** into `/kaggle/working/checkpoints`, so you can download each batch directly from the Kaggle output panel. **No Google Drive upload.**
6. Cell 6 (**assembly**) merges only the **complete** 1000-person checkpoints into a single downloadable `final_dataset.zip` - use it whenever you stop the run early.

**Before running:** Settings -> Internet: ON, Accelerator: GPU T4 x2. The Drive file must be shared **"Anyone with the link"**. Run cells top to bottom - no kernel restart needed.

In [1]:
# 1) Install requirements. facenet-pytorch with --no-deps so it does NOT disturb Kaggle's working GPU torch.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "facenet-pytorch"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "gdown"], check=False)
print("install done")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 16.8 MB/s eta 0:00:00
install done


In [2]:
# 2) GPU check - PyTorch sees the T4s with zero setup.
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  ", i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), "No GPU - set Accelerator to GPU T4 x2 in Settings."


torch: 2.10.0+cu128
CUDA available: True
GPU count: 2
   0 Tesla T4
   1 Tesla T4


In [3]:
# 3) Download + extract dataset from Google Drive by id (robust for large public files).
#    Skips the ~15 GB download if it is already extracted.
import gdown, os, zipfile, shutil

DATA_FILE_ID = "1llJJIM-pyHU90hPVTV0oLQYtGb4UBhUI"   # <-- your data.zip file id
DATA_PARENT  = "/kaggle/tmp/data"
DATA_ROOT    = "/kaggle/tmp/data/ms1m-arcface"

def _looks_extracted(path):
    return os.path.isdir(path) and sum(
        os.path.isdir(os.path.join(path, d)) for d in os.listdir(path)) > 1000

if _looks_extracted(DATA_ROOT):
    print("Already extracted:", len(os.listdir(DATA_ROOT)), "identities - skipping download")
else:
    os.makedirs("/kaggle/tmp", exist_ok=True)
    zip_path = "/kaggle/tmp/data.zip"

    if not os.path.exists(zip_path):
        got = None
        # Try 1: id form (handles the large-file confirmation page in recent gdown)
        try:
            got = gdown.download(id=DATA_FILE_ID, output=zip_path, quiet=False, resume=True)
        except Exception as e:
            print("gdown id= attempt failed:", e)
        # Try 2: fuzzy URL form as a fallback
        if not got:
            try:
                url = f"https://drive.google.com/uc?id={DATA_FILE_ID}"
                got = gdown.download(url, zip_path, quiet=False, fuzzy=True, resume=True)
            except Exception as e:
                print("gdown fuzzy attempt failed:", e)
        if not got or not os.path.exists(zip_path):
            raise RuntimeError(
                "Download failed. Make sure the Drive file is shared 'Anyone with the link' "
                "and is not blocked by Google's download quota, then re-run this cell.")

    free_gb = shutil.disk_usage("/kaggle/tmp").free / (1024**3)
    print(f"free space: {free_gb:.1f} GB - extracting...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_PARENT)
    os.remove(zip_path)   # free ~15 GB

    # if the zip contained a single top folder, descend into it
    root = DATA_PARENT
    subs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    if len(subs) == 1:
        root = os.path.join(root, subs[0])
    DATA_ROOT = root

print("Data root:", DATA_ROOT, "| identities:", len(os.listdir(DATA_ROOT)))


Downloading...
From (original): https://drive.google.com/uc?id=1llJJIM-pyHU90hPVTV0oLQYtGb4UBhUI
From (redirected): https://drive.google.com/uc?id=1llJJIM-pyHU90hPVTV0oLQYtGb4UBhUI&confirm=t&uuid=64265ea1-3f84-45da-8a61-90b2ad819d95
To: /kaggle/tmp/data.zip
100%|██████████| 16.6G/16.6G [02:16<00:00, 122MB/s] 


free space: 1048.1 GB - extracting...
Data root: /kaggle/tmp/data/ms1m-arcface | identities: 85742


In [4]:
%%writefile /kaggle/working/face_clean.py
# Detection / scoring / matching / worker module - imported by the spawn'd GPU processes in the next cell.
import os, glob, re, shutil, math, traceback, zipfile
import cv2, numpy as np
import torch
import torch.nn.functional as F

def metrics_from_landmarks(lmk, bbox_area, prob):
    # facenet-pytorch MTCNN landmark order: left_eye, right_eye, nose, mouth_left, mouth_right
    le, re_, nose, ml, mr = [np.asarray(p, dtype=np.float32) for p in lmk]
    iod = float(np.linalg.norm(re_ - le)) + 1e-6
    eye_mid = (le + re_) / 2.0
    eye_dir = (re_ - le) / iod
    roll = abs(math.degrees(math.atan2(float(re_[1]-le[1]), float(re_[0]-le[0]))))
    yaw_proxy = abs(float(np.dot(nose - eye_mid, eye_dir))) / iod
    mouth_ratio = float(np.linalg.norm(mr - ml)) / iod
    return {'roll': roll, 'yaw': yaw_proxy, 'mouth': mouth_ratio,
            'face_px': float(bbox_area), 'det': float(prob)}

def quality_score(m, sharp):
    # single unified score used to pick BOTH reference and match:
    # resolution + frontal-pose + neutrality + uprightness + sharpness (+ detection confidence)
    frontal = math.exp(-(m['yaw']/0.20)**2)
    upright = math.exp(-(m['roll']/12.0)**2)
    neutral = math.exp(-(max(0.0, m['mouth']-0.95)/0.35)**2)
    sh  = min(1.0, sharp/120.0)
    res = min(1.0, m['face_px']/(112.0*112.0))
    return 0.25*res + 0.25*frontal + 0.20*sh + 0.15*upright + 0.10*neutral + 0.05*m['det']

def _face_tensor(bgr_crop):
    # -> (3,160,160) float tensor, standardized the way facenet-pytorch expects
    rgb = cv2.cvtColor(bgr_crop, cv2.COLOR_BGR2RGB)
    rgb = cv2.resize(rgb, (160, 160), interpolation=cv2.INTER_AREA)
    t = torch.from_numpy(rgb).float().permute(2, 0, 1)
    return (t - 127.5) / 128.0

@torch.no_grad()
def cosine_same_person(resnet, device, crop_a, crop_b):
    # embed both faces in one forward pass, return cosine similarity in [-1, 1]
    batch = torch.stack([_face_tensor(crop_a), _face_tensor(crop_b)]).to(device)
    emb = F.normalize(resnet(batch), dim=1)
    return float((emb[0] * emb[1]).sum())

def process_identity(mtcnn, resnet, device, ident_dir, out_dir, match_thr):
    cands = []
    for fp in glob.glob(os.path.join(ident_dir, '*')):
        img = cv2.imread(fp)
        if img is None:
            continue
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        boxes, probs, lms = mtcnn.detect(rgb, landmarks=True)
        if boxes is None or lms is None:
            continue
        areas = [float((b[2]-b[0])*(b[3]-b[1])) for b in boxes]
        i = int(np.argmax(areas))
        if probs[i] is None or float(probs[i]) < 0.90:
            continue
        x1, y1, x2, y2 = [int(round(v)) for v in boxes[i]]
        x1, y1 = max(0, x1), max(0, y1)
        crop = img[y1:y2, x1:x2]
        if crop.size == 0:
            continue
        gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        sharp = float(cv2.Laplacian(gray, cv2.CV_64F).var())
        m = metrics_from_landmarks(lms[i], areas[i], probs[i])
        score = quality_score(m, sharp)
        cands.append({'src': fp, 'score': score, 'crop': crop})
    if len(cands) < 2:
        return False

    # rank all candidates by the SAME quality_score: 1st qualified -> reference, 2nd -> match
    ranked = sorted(cands, key=lambda c: c['score'], reverse=True)
    ref, mat = ranked[0], ranked[1]

    # --- identity match check: keep the person ONLY if reference & match are the same face ---
    sim = cosine_same_person(resnet, device, ref['crop'], mat['crop'])
    if sim < match_thr:
        return False   # unmatched -> dropped, NOT counted toward the 1000

    os.makedirs(out_dir, exist_ok=True)
    re_ext = os.path.splitext(ref['src'])[1] or '.jpg'
    ma_ext = os.path.splitext(mat['src'])[1] or '.jpg'
    # save the FULL original images (byte-exact copy, no crop / no re-encode)
    shutil.copy2(ref['src'], os.path.join(out_dir, 'reference' + re_ext))
    shutil.copy2(mat['src'], os.path.join(out_dir, 'match' + ma_ext))
    return True

def _zip_batch(out_root, ident_batch, ckpt_dir, zip_name, cleanup=True):
    os.makedirs(ckpt_dir, exist_ok=True)
    zip_path = os.path.join(ckpt_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:  # jpgs already compressed
        for ident in ident_batch:
            d = os.path.join(out_root, ident)
            if not os.path.isdir(d):
                continue
            for fn in os.listdir(d):
                z.write(os.path.join(d, fn), arcname=os.path.join(ident, fn))
    if cleanup:  # free /kaggle/working so it stays small - the zip is the deliverable
        for ident in ident_batch:
            shutil.rmtree(os.path.join(out_root, ident), ignore_errors=True)
    return zip_path

def worker(gpu_id, ident_list, data_root, out_root, ckpt, ckpt_dir, ckpt_every, match_thr):
    from facenet_pytorch import MTCNN, InceptionResnetV1
    device = f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu'
    mtcnn = MTCNN(keep_all=True, post_process=False, device=device)
    resnet = InceptionResnetV1(pretrained='vggface2').eval().to(device)

    # resume-safe: continue zip numbering after any zips left from a previous run
    existing = glob.glob(os.path.join(ckpt_dir, f"gpu{gpu_id}_batch*.zip"))
    nums = [int(re.search(r'batch(\d+)', os.path.basename(p)).group(1))
            for p in existing if re.search(r'batch(\d+)', os.path.basename(p))]
    batch_no = max(nums) if nums else 0

    batch, kept, dropped = [], 0, 0
    ck = open(f'{ckpt}.gpu{gpu_id}', 'a')
    try:
        total = len(ident_list)
        for n, ident in enumerate(ident_list, 1):
            try:
                ok = process_identity(mtcnn, resnet, device,
                                      os.path.join(data_root, ident),
                                      os.path.join(out_root, ident), match_thr)
            except Exception:
                traceback.print_exc()
                ok = False
            ck.write(ident + "\n"); ck.flush()   # record as processed (for resume)
            if ok:
                batch.append(ident); kept += 1
            else:
                dropped += 1
            if len(batch) >= ckpt_every:            # 1000 MATCHED identities -> one zip
                batch_no += 1
                zp = _zip_batch(out_root, batch, ckpt_dir, f"gpu{gpu_id}_batch{batch_no:04d}.zip")
                print(f"[gpu{gpu_id}] saved {zp} | {len(batch)} identities | processed {n}/{total} | dropped {dropped}", flush=True)
                batch = []
        if batch:                                   # final partial batch (< ckpt_every)
            batch_no += 1
            zp = _zip_batch(out_root, batch, ckpt_dir, f"gpu{gpu_id}_batch{batch_no:04d}.zip")
            print(f"[gpu{gpu_id}] saved final {zp} | {len(batch)} identities (PARTIAL)", flush=True)
    finally:
        ck.close()
    print(f"[gpu{gpu_id}] DONE - kept {kept}, dropped {dropped}, across {batch_no} zips", flush=True)


Writing /kaggle/working/face_clean.py


In [5]:
# 5) Run on both T4s (one spawn process per GPU). Saves a zip every CHECKPOINT_EVERY MATCHED
#    identities PER GPU into /kaggle/working/checkpoints - download those from the output panel.
#    Each kept identity is verified: reference & match must be the SAME person
#    (cosine similarity >= MATCH_THRESHOLD) or the identity is dropped.
import os, glob, sys, importlib
import torch, torch.multiprocessing as tmp

sys.path.insert(0, "/kaggle/working")
import face_clean; importlib.reload(face_clean)

OUT_ROOT = "/kaggle/working/cleaned"
CKPT = "/kaggle/working/done_ids.txt"
CKPT_DIR = "/kaggle/working/checkpoints"
CHECKPOINT_EVERY = 1000     # zip every N MATCHED identities, PER GPU worker
MATCH_THRESHOLD  = 0.50     # cosine similarity; below this, reference/match count as different people -> drop
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

# pre-download model weights once in the parent so the two GPU workers don't race the cache
from facenet_pytorch import MTCNN, InceptionResnetV1
_warm_a = MTCNN(keep_all=True, post_process=False, device='cpu')
_warm_b = InceptionResnetV1(pretrained='vggface2').eval()
del _warm_a, _warm_b
print("model weights ready")

done = set()
for fpath in glob.glob(CKPT + ".gpu*"):
    done.update(open(fpath).read().split())
all_ids = sorted(d for d in os.listdir(DATA_ROOT)
                 if os.path.isdir(os.path.join(DATA_ROOT, d)) and d not in done)
print(f"{len(all_ids)} identities remaining, {len(done)} already done")

ngpu = max(1, torch.cuda.device_count())
chunks = [all_ids[g::ngpu] for g in range(ngpu)]

ctx = tmp.get_context("spawn")
procs = []
for g in range(ngpu):
    p = ctx.Process(target=face_clean.worker,
                    args=(g, chunks[g], DATA_ROOT, OUT_ROOT, CKPT, CKPT_DIR, CHECKPOINT_EVERY, MATCH_THRESHOLD))
    p.start(); procs.append(p)
for p in procs:
    p.join()

print("Finished. Checkpoint zips in:", CKPT_DIR)
print("Files:", sorted(os.listdir(CKPT_DIR)))


  0%|          | 0.00/107M [00:00<?, ?B/s]

model weights ready
85742 identities remaining, 0 already done
[gpu0] saved /kaggle/working/checkpoints/gpu0_batch0001.zip | 1000 identities | processed 1026/42871 | dropped 26
[gpu1] saved /kaggle/working/checkpoints/gpu1_batch0001.zip | 1000 identities | processed 1019/42871 | dropped 19
[gpu0] saved /kaggle/working/checkpoints/gpu0_batch0002.zip | 1000 identities | processed 2123/42871 | dropped 123
[gpu1] saved /kaggle/working/checkpoints/gpu1_batch0002.zip | 1000 identities | processed 2129/42871 | dropped 129
[gpu0] saved /kaggle/working/checkpoints/gpu0_batch0003.zip | 1000 identities | processed 3232/42871 | dropped 232
[gpu1] saved /kaggle/working/checkpoints/gpu1_batch0003.zip | 1000 identities | processed 3256/42871 | dropped 256
[gpu0] saved /kaggle/working/checkpoints/gpu0_batch0004.zip | 1000 identities | processed 4346/42871 | dropped 346
[gpu1] saved /kaggle/working/checkpoints/gpu1_batch0004.zip | 1000 identities | processed 4380/42871 | dropped 380
[gpu0] saved /kaggl

Process SpawnProcess-2:
Process SpawnProcess-1:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/kaggle/working/face_clean.py", line 120, in worker
    ok = process_identity(mtcnn, resnet, device,
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/face_clean.py", line 50, in process_identity
    boxes, probs, lms = mtcnn.detect(rgb, landmarks=True)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/facenet_pytorch/models/mtcnn.py", line 313, in detect
    batch_boxes, batch_points = detect_face(
                                ^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/facenet_pytorch/models/utils/detect_face.py", line 45, in detect_face
    model_dtype = next(pnet.parameters()).dtype


KeyboardInterrupt: 

In [6]:
# 6) Assemble the COMPLETE checkpoints (exactly CHECKPOINT_EVERY persons each) into one
#    downloadable zip. Run this whenever you stop the cleaning cell early: it ignores any
#    partial checkpoint and merges only the full 1000-person batches. Each person subfolder
#    keeps its reference + match images.
import os, glob, zipfile

CKPT_DIR   = "/kaggle/working/checkpoints"
BATCH_SIZE = 1000                                   # a "complete" checkpoint holds exactly this many persons
FINAL_ZIP  = "/kaggle/working/final_dataset.zip"

def _persons_in_zip(zpath):
    with zipfile.ZipFile(zpath) as z:
        tops = set()
        for name in z.namelist():
            if name.endswith('/'):
                continue
            tops.add(name.split('/')[0])
    return tops

batch_zips = sorted(glob.glob(os.path.join(CKPT_DIR, "gpu*_batch*.zip")))
print(f"Found {len(batch_zips)} checkpoint zip(s) in {CKPT_DIR}")

complete = []
for zp in batch_zips:
    n_persons = len(_persons_in_zip(zp))
    if n_persons == BATCH_SIZE:
        complete.append(zp)
        print(f"  include  {os.path.basename(zp)}  ({n_persons} persons)")
    else:
        print(f"  skip     {os.path.basename(zp)}  ({n_persons} persons - not a complete {BATCH_SIZE})")

if not complete:
    print("\nNo complete checkpoints found - nothing to assemble.")
else:
    seen = set()
    with zipfile.ZipFile(FINAL_ZIP, "w", zipfile.ZIP_STORED) as out:   # jpgs already compressed
        for zp in complete:
            with zipfile.ZipFile(zp) as z:
                for name in z.namelist():
                    if name.endswith('/') or name in seen:
                        continue
                    seen.add(name)
                    out.writestr(name, z.read(name))

    persons = {n.split('/')[0] for n in seen}
    size_gb = os.path.getsize(FINAL_ZIP) / (1024**3)
    print(f"\nAssembled {len(complete)} checkpoint(s) -> {FINAL_ZIP}")
    print(f"{len(persons)} persons | {len(seen)} images | {size_gb:.2f} GB")
    print("Download final_dataset.zip from the Kaggle output panel (Data / Output).")


Found 14 checkpoint zip(s) in /kaggle/working/checkpoints
  include  gpu0_batch0001.zip  (1000 persons)
  include  gpu0_batch0002.zip  (1000 persons)
  include  gpu0_batch0003.zip  (1000 persons)
  include  gpu0_batch0004.zip  (1000 persons)
  include  gpu0_batch0005.zip  (1000 persons)
  include  gpu0_batch0006.zip  (1000 persons)
  include  gpu0_batch0007.zip  (1000 persons)
  include  gpu1_batch0001.zip  (1000 persons)
  include  gpu1_batch0002.zip  (1000 persons)
  include  gpu1_batch0003.zip  (1000 persons)
  include  gpu1_batch0004.zip  (1000 persons)
  include  gpu1_batch0005.zip  (1000 persons)
  include  gpu1_batch0006.zip  (1000 persons)
  include  gpu1_batch0007.zip  (1000 persons)

Assembled 14 checkpoint(s) -> /kaggle/working/final_dataset.zip
14000 persons | 28000 images | 0.08 GB
Download final_dataset.zip from the Kaggle output panel (Data / Output).
